In [13]:
from pyspark.sql import SparkSession
from itertools import combinations
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("MarketBasket") \
    .master("local[*]") \
    .getOrCreate()

df = spark.read.csv("/home/jovyan/work/ecommerce_logs.csv", header=True, inferSchema=True)

print(df.count())
df.show(5)

10831817
+-------------------+---------------+---------+----------+----------+------+-------------+--------------------+--------------------+
|          timestamp|     session_id|  user_id|event_type|product_id| price|     referrer|       user_metadata|    product_metadata|
+-------------------+---------------+---------+----------+----------+------+-------------+--------------------+--------------------+
|2026-02-21 08:33:36|SESS_4007a38849|User_2686|      view| ITEM_3306| 754.5|instagram.com|"{""device"": ""M...|  ""tier"": ""Gold""|
|2026-02-21 08:34:02|SESS_4007a38849|User_2686|      cart| ITEM_4021|654.44|instagram.com|"{""device"": ""M...|  ""tier"": ""Gold""|
|2026-02-21 08:35:42|SESS_4007a38849|User_2686|      view| ITEM_1241|608.71|instagram.com|"{""device"": ""M...|  ""tier"": ""Gold""|
|2026-02-21 08:41:33|SESS_4007a38849|User_2686|      view| ITEM_4064|396.02|instagram.com|                NULL|                NULL|
|2026-02-21 08:43:56|SESS_4007a38849|User_2686|      view| I

In [18]:
# 1. Filter purchases
purchases = df.filter(df.event_type == "purchase")
clean = purchases.filter(
    col("session_id").isNotNull() &
    col("product_id").isNotNull()
)

In [19]:
# 2. Map step
rdd = clean.rdd.map(lambda row: (row.session_id, row.product_id))

In [20]:
# 3. Group by session
grouped = rdd.groupByKey()

In [32]:
# 4. Generate pairs
pairs = grouped.flatMap(
    lambda x: [((i, j), 1) for i, j in combinations(set(x[1]), 2)]
)

In [33]:
# 5. Reduce step
pair_counts = pairs.reduceByKey(lambda a, b: a + b)

In [34]:
# 6. Sort results
sorted_pairs = pair_counts.sortBy(lambda x: x[1], ascending=False)

In [35]:
# 7. Show output
print(sorted_pairs.take(20))

[(('ITEM_3719', 'ITEM_3134'), 2), (('ITEM_2468', 'ITEM_376'), 2), (('ITEM_3647', 'ITEM_993'), 2), (('ITEM_3469', 'ITEM_1152'), 2), (('ITEM_4440', 'ITEM_57'), 2), (('ITEM_3832', 'ITEM_276'), 2), (('ITEM_1818', 'ITEM_4049'), 2), (('ITEM_61', 'ITEM_4597'), 2), (('ITEM_3542', 'ITEM_1526'), 2), (('ITEM_2097', 'ITEM_292'), 2), (('ITEM_1691', 'ITEM_2491'), 2), (('ITEM_148', 'ITEM_2945'), 2), (('ITEM_3031', 'ITEM_3576'), 2), (('ITEM_106', 'ITEM_1349'), 2), (('ITEM_676', 'ITEM_886'), 2), (('ITEM_122', 'ITEM_2089'), 2), (('ITEM_1166', 'ITEM_4307'), 2), (('ITEM_26', 'ITEM_3725'), 2), (('ITEM_4052', 'ITEM_3532'), 2), (('ITEM_2355', 'ITEM_4236'), 2)]


In [37]:
# 8. Save output
sorted_pairs.saveAsTextFile("output/market_basket")